# Ⅰ第1回 演習1（E1-1）環境ベンチマーク

**今日の問い**：良いAIとは何を測ることか

この演習では自分のマシンの「機種・速度・メモリ」を測り，グループの台帳（CSV）に投稿する．
ブロックを上から順に実行し，`TODO` の箇所を埋める．実行時間の目安は 3 分以内．

**この演習で身につけること**
- GPU（MPS）は非同期に動く．同期を取らなければ時間計測は正しくない
- 「速い」には 2 種類ある：メモリの読み書きで速さが決まる処理と，計算の量で速さが決まる処理．どちらになるかは処理で変わる


## (0) グループと役割の設定

In [ ]:
# ===== (0) グループと役割の設定 =====
GROUP_ID = 1                  # ← 自分のグループ番号 (1〜27) に書き換える
MEMBER_ROLE = "implementer"   # implementer / verifier / recorder / presenter（5 人グループは collector も）のいずれか．3 人グループで presenter を兼ねる verifier は "verifier"

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

## (1) コードテンプレート①：ライブラリのインポート

教科書 2.3 節のテンプレート①に，授業用の計測モジュール（`common`）を加えたもの．

In [ ]:
import time                       # 時間計測
import json
import numpy as np
import torch                      # PyTorch
import pandas as pd
from common.device import get_device, synchronize, machine_info   # MPS/CPU 選択と同期
from common import memory, bench                                   # ピークメモリ計測・ベンチマーク
from common.logger import ResultLogger                             # 台帳（CSV）への記録
print("torch", torch.__version__)

## (2) 計測：機種情報の取得

チップ名・メモリ量・OS・PyTorch の版を取る．この行がそのまま台帳の1行になる．

In [ ]:
info = machine_info()
for k, v in info.items():
    print(f"{k:14s}: {v}")

## (3) コードテンプレート⑤：使用デバイスの選択

教科書 2.3 節のテンプレート⑤と同じ 3 分岐（cuda → mps → cpu）．受講者の MacBook Air では MPS（Apple GPU）が選ばれる．
授業用の `get_device()` は⑤に，16GB 機の実使用可能量に合わせた **11GB のメモリ上限** を加えたもの．

In [ ]:
# コードテンプレート⑤
# TODO: MPS（Apple GPU）が使えるかを判定する条件を書く（torch.backends.mps.is_available() を使う）
MPS_AVAILABLE = ...
assert MPS_AVAILABLE is not ..., "TODO 未実装: MPS が使えるかの判定を書く"
if torch.cuda.is_available():
    device = torch.device("cuda")
elif MPS_AVAILABLE:
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"使用デバイス: {device}")

device = get_device()      # 授業では以降これを使う（⑤ ＋ メモリ上限）
print("使用デバイス:", device, "| メモリ上限:", memory.cap_torch())

## (4) 計測：同期しないと何が起きるか

GPU は命令を受け取るとすぐ制御を返し，裏で計算を続ける．
`time.perf_counter()` で囲むだけでは **命令を送った時間** しか測れない．
計測の前後で `synchronize(device)` を呼び，計算の完了を待ってから時計を読む．

In [ ]:
N = 2048
a = torch.randn(N, N, device=device)
b = torch.randn(N, N, device=device)
_ = a @ b                                    # ウォームアップ（初回はカーネルのコンパイルで遅い）

# --- 同期なし ---
t0 = time.perf_counter()
for _ in range(10):
    c = a @ b
t_nosync = (time.perf_counter() - t0) / 10

# --- 同期あり ---
# TODO: 計測の「前」と「後」に synchronize(device) を入れる
t0 = time.perf_counter()
for _ in range(10):
    c = a @ b
t_sync = (time.perf_counter() - t0) / 10

print(f"同期なし: {t_nosync*1000:8.3f} ms / 回")
print(f"同期あり: {t_sync*1000:8.3f} ms / 回")
print(f"比: {t_sync / max(t_nosync, 1e-9):.1f} 倍   ← MPS では大きく異なる．CPU ではほぼ 1")

## (5) 計測 B1：メモリ帯域（GB/s）

大きなベクトルの加算 `c = a + b` は計算そのものはすぐ終わり，**メモリからデータを読み書きする時間** で全体の速さが決まる（ボトルネックになる）．
1回の加算で動くバイト数は「a を読む + b を読む + c を書く」= 3 × 要素数 × 4 バイト（float32）．

**もう 1 つの落とし穴**：GPU は最初の実行に準備の時間が入るので，最初の数回は捨てる（ウォームアップ）．
1 回だけの値はぶれるので，**2 秒ほど回して中央値** を取る．

In [ ]:
def timed_loop(fn, min_sec=2.0, min_iters=5):
    """fn を最低 min_sec 秒回し，1 回あたりの時間の中央値を返す"""
    for _ in range(2):
        fn()                                  # ウォームアップ
    synchronize(device)
    times = []
    t_start = time.perf_counter()
    while len(times) < min_iters or time.perf_counter() - t_start < min_sec:
        t0 = time.perf_counter()
        fn()
        synchronize(device)                   # 計算の完了を待ってから時計を読む
        times.append(time.perf_counter() - t0)
    return float(np.median(times))

n = 64 * 1024 * 1024                          # 6400万要素（256MB × 3）
a = torch.ones(n, device=device)
b = torch.ones(n, device=device)
sec = timed_loop(lambda: torch.add(a, b))

# TODO: 1回の加算で動くバイト数を式で書く（読み2本 + 書き1本，float32 = 4 バイト）
bytes_moved = ...
bandwidth_gbps = bytes_moved / sec / 1e9
print(f"B1 帯域: {bandwidth_gbps:.1f} GB/s  ({sec*1000:.2f} ms/回，中央値)")
nominal = bench.nominal_bandwidth_gbps(info["chip"])
if nominal:
    print(f"   公称値 {nominal:g} GB/s（{info['chip']}）の {bandwidth_gbps / nominal * 100:.0f}%   ← 100% を超えたら式（TODO 3）を疑う．同じチップの人と比べる")
else:
    print("   この機種の公称値は表に無い（授業ページの表を見る）")

## (6) 計測 B2：演算性能（GFLOPS）

N×N の行列積は 2N³ 回の浮動小数点演算（乗算と加算）を行う．こちらは読み書きより **計算の量** で全体の速さが決まる．

In [ ]:
N = 2048
a = torch.randn(N, N, device=device)
b = torch.randn(N, N, device=device)
sec = timed_loop(lambda: a @ b)

# TODO: N×N 行列積 1 回の浮動小数点演算数を式で書く
flops = ...
gflops = flops / sec / 1e9
print(f"B2 演算: {gflops:.0f} GFLOPS  ({sec*1000:.2f} ms/回，中央値)")

## (7) 計測 B3：学習ステップ時間（ms/step）

授業で使う小型 Transformer（A3，約30万パラメータ）の 1 ステップ（順伝播＋逆伝播＋更新）にかかる時間．
これが授業中の「待ち時間」を決める．

In [ ]:
step_ms, n_params = bench.train_step_ms(device)
print(f"B3 学習: {step_ms:.2f} ms/step  (モデル {n_params:,} パラメータ, バッチ 128)")

## (8) 計測：メモリ

1GB のテンソルを確保したときのピークを測る．`PeakSampler` はブロック内の最大使用量を裏で採取する．

In [ ]:
with memory.PeakSampler("driver") as ps:             # MPS ドライバが確保した量（CPU 時は RSS）
    big = torch.empty(256 * 1024 * 1024, device=device)  # 1GB
    big.fill_(1.0)
    synchronize(device)
del big
print(f"ピーク: {ps.peak_mb:.0f} MB  増分: {ps.delta_mb:.0f} MB  種別: {ps.kind}")
print("この機の搭載メモリ:", info["memory_gb"], "GB")

## (9) 記録：台帳へ投稿

測った値を共通形式の CSV に追記する．グループの全員が投稿し，`aggregate/c1d1_env.py` で提出された分を1枚にまとめる．

In [ ]:
logger = ResultLogger(GROUP_ID, MEMBER_ROLE, course="c1", day="d1", exercise="ex1", device=device)
logger.log_many({
    "bandwidth_gbps": bandwidth_gbps,
    "matmul_gflops": gflops,
    "train_step_ms": step_ms,
    "memory_gb": info["memory_gb"],
    "alloc_1gb_peak_mb": ps.peak_mb,
    "sync_ratio": t_sync / max(t_nosync, 1e-9),
}, condition=info["chip"] or "unknown")
print("書き込み先:", logger.path)
# 台帳に貼る 1 行
print(f"{logger.machine} | {device} | B1 {bandwidth_gbps:.0f} GB/s | B2 {gflops:.0f} GFLOPS | B3 {step_ms:.1f} ms/step")

## (10) 記録の確認

自分が書いた CSV を読み戻す．この形式が全14回で共通になる．

In [ ]:
df = pd.read_csv(logger.path)
df.tail(6)

## (11) グループディスカッション（5 分）→ グループ内プレゼン1（1 人 2 分）

presenter が進行し，グループの 4 台の数値を並べて **全員が 1 回は発言** する．recorder は結論を 3 行で `results/` の失敗記録に残す．

**討議の問い**（順に）
1. 同期あり／なしの比はグループ内で何倍から何倍まであったか．なぜ機種で違うか
2. 帯域は公称値の何 % 出たか．待ち時間用課題に書いた予測と比べてどうか
3. B1（帯域）と B2（演算）で，グループ内の機種差が大きいのはどちらか．その理由は

**プレゼン1 の型**：presenter がグループの結論を 2 分 → implementer・verifier・recorder が 1 分ずつ補足（`README.md` の役割別テーマ）

In [ ]:
# 討議用にグループの 4 台ぶんを並べる（各自の CSV が同じフォルダにある場合）．無い場合は自分の 1 行だけ出る．
df_all = pd.read_csv(logger.path)
latest = df_all.sort_values("timestamp").groupby(["machine", "metric_name"]).tail(1)
board = latest.pivot_table(index="machine", columns="metric_name", values="metric_value")
cols = [c for c in ["sync_ratio", "bandwidth_gbps", "matmul_gflops", "train_step_ms", "memory_gb"] if c in board]
display(board[cols].round(1))
print("討議メモ（recorder が埋める）:")
print("  1. 同期比の範囲: ____ 倍 〜 ____ 倍  理由: ________")
print("  2. 帯域は公称の ____ %  予測との差: ________")
print("  3. 機種差が大きいのは B__  理由: ________")